In [0]:
from pyspark.sql.functions import current_timestamp

dbutils.widgets.text("catalog_name", "cs2_dev")
catalog_name = dbutils.widgets.get("catalog_name")

dbutils.widgets.text("storage_account", "datalakecstracker")
storage_account = dbutils.widgets.get("storage_account")

dbutils.widgets.text("secret_scope", "cs2tracker-secrets")
secret_scope = dbutils.widgets.get("secret_scope")
conn_string = dbutils.secrets.get(scope=secret_scope, key="evh-connection-string")


dbutils.widgets.text("evh_namespace", "evhcs2tracker")
evh_namespace = dbutils.widgets.get("evh_namespace")

dbutils.widgets.text("evh_name", "cs2tracker_evh")
evh_name = dbutils.widgets.get("evh_name")

In [0]:
table_path = f"abfss://bronze@{storage_account}.dfs.core.windows.net/tables/gsi_streaming_data"
checkpoint_path = f"abfss://bronze@{storage_account}.dfs.core.windows.net/checkpoints/gsi_streaming_data"

#Reading data from Event Hub
df_stream = (spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", f"{evh_namespace}.servicebus.windows.net:9093")
    .option("subscribe", evh_name)
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.jaas.config", f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="{conn_string}";')
    .option("startingOffsets", "earliest")
    .load()
)

df_bronze = df_stream.withColumn("ingest_timestamp", current_timestamp())

In [0]:
(df_bronze.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .option("path", table_path)
    .trigger(availableNow=True)
    .toTable(f"{catalog_name}.bronze.gsi_raw_data")
)
#POPRAWIC EVENT HUB Z BASIC NA STANDARD